In [1]:
# 1. SigmaPPG 설치
!git clone https://github.com/ZonghengGuo/SigmaPPG.git
%cd /content/SigmaPPG
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# 2. Imports and configuration
import csv
import gc
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn

from einops import rearrange
from scipy.signal import resample_poly
from torch.utils.data import Dataset, DataLoader
from downstream.model_select import select_model


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "datasets/PulseDB/Normal"
)

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Mine/SIGMA-PPG/sigma.pth"
)

ORIGINAL_FS = 125
TARGET_FS = 100
PATCH_SIZE = 50

TRAIN_SUBJECT_RATIO = 0.80
SEED = 42

BATCH_SIZE = 32
EPOCHS = 150
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

print("Device:", DEVICE)
print("Data directory:", DATA_DIR)

Device: cuda
Data directory: /content/drive/MyDrive/Colab Notebooks/datasets/PulseDB/Normal


In [4]:
# 3. Utilities and PulseDB loader
def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_matlab_reference(file, reference):
    return np.asarray(file[reference][()]).squeeze()


def resample_ppg(ppg, original_fs=ORIGINAL_FS, target_fs=TARGET_FS):
    ppg = np.asarray(ppg, dtype=np.float32).reshape(-1)

    gcd = np.gcd(original_fs, target_fs)

    return resample_poly(
        ppg,
        up=target_fs // gcd,
        down=original_fs // gcd,
    ).astype(np.float32)


def load_pulsedb_subject(mat_file):
    ppg_list = []
    label_list = []

    with h5py.File(mat_file, "r") as file:
        group = file["Subj_Wins"]

        ppg_refs = group["PPG_F"][0]
        sbp_refs = group["SegSBP"][0]
        dbp_refs = group["SegDBP"][0]

        for ppg_ref, sbp_ref, dbp_ref in zip(
            ppg_refs,
            sbp_refs,
            dbp_refs,
        ):
            ppg = read_matlab_reference(file, ppg_ref)
            sbp = float(read_matlab_reference(file, sbp_ref))
            dbp = float(read_matlab_reference(file, dbp_ref))

            ppg = np.asarray(ppg, dtype=np.float32).reshape(-1)

            if len(ppg) == 0:
                continue
            if not np.all(np.isfinite(ppg)):
                continue
            if not np.isfinite(sbp) or not np.isfinite(dbp):
                continue
            if sbp <= dbp:
                continue

            ppg_list.append(resample_ppg(ppg))
            label_list.append([sbp, dbp])

    if not ppg_list:
        raise ValueError("No valid PPG segments were found.")

    X = np.stack(ppg_list).astype(np.float32)[:, None, :]
    y = np.asarray(label_list, dtype=np.float32)

    return X, y


class PulseDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [5]:
# 4. Subject-wise split
set_seed(SEED)

mat_files = sorted(DATA_DIR.glob("*.mat"))

if not mat_files:
    raise FileNotFoundError(
        f"No .mat files were found in: {DATA_DIR}"
    )

rng = np.random.default_rng(SEED)
subject_indices = rng.permutation(len(mat_files))

num_train_subjects = int(
    len(mat_files) * TRAIN_SUBJECT_RATIO
)

train_indices = subject_indices[:num_train_subjects]
test_indices = subject_indices[num_train_subjects:]

train_files = [mat_files[index] for index in train_indices]
test_files = [mat_files[index] for index in test_indices]

if not train_files or not test_files:
    raise ValueError(
        "Both train and test subject groups must be non-empty."
    )

print("=" * 70)
print("Total subjects:", len(mat_files))
print("Train subjects:", len(train_files))
print("Test subjects :", len(test_files))
print("=" * 70)

print("\nTrain subject examples:")
for file in train_files[:5]:
    print("-", file.name)

print("\nTest subjects:")
for file in test_files:
    print("-", file.name)

Total subjects: 10
Train subjects: 8
Test subjects : 2

Train subject examples:
- p059290.mat
- p062917.mat
- p004679.mat
- p081410.mat
- p030670.mat

Test subjects:
- p017795.mat
- p098382.mat


In [6]:
# 5. Compute label normalization using train subjects only
label_sum = np.zeros(2, dtype=np.float64)
label_squared_sum = np.zeros(2, dtype=np.float64)
label_count = 0

input_size = None
valid_train_files = []

for index, mat_file in enumerate(train_files, start=1):
    try:
        X_subject, y_subject = load_pulsedb_subject(mat_file)

        if input_size is None:
            input_size = X_subject.shape[-1]

        if X_subject.shape[-1] != input_size:
            raise ValueError(
                f"Inconsistent input length: "
                f"{X_subject.shape[-1]} != {input_size}"
            )

        label_sum += y_subject.sum(axis=0)
        label_squared_sum += (y_subject ** 2).sum(axis=0)
        label_count += len(y_subject)

        valid_train_files.append(mat_file)

        print(
            f"[{index:03d}/{len(train_files):03d}] "
            f"{mat_file.name}: {len(y_subject)} segments"
        )

        del X_subject, y_subject
        gc.collect()

    except Exception as error:
        print(f"Skipped {mat_file.name}: {error}")


if label_count == 0:
    raise RuntimeError("No valid training labels were found.")

y_mean = label_sum / label_count
y_variance = (
    label_squared_sum / label_count
    - y_mean ** 2
)
y_std = np.sqrt(np.maximum(y_variance, 1e-8))

y_mean = y_mean.reshape(1, 2).astype(np.float32)
y_std = y_std.reshape(1, 2).astype(np.float32)

train_files = valid_train_files

print("\n" + "=" * 70)
print("Training segments:", label_count)
print("Input size:", input_size)
print("Train label mean [SBP, DBP]:", y_mean.squeeze())
print("Train label std  [SBP, DBP]:", y_std.squeeze())
print("=" * 70)

[001/008] p059290.mat: 2053 segments
[002/008] p062917.mat: 1743 segments
[003/008] p004679.mat: 1862 segments
[004/008] p081410.mat: 1741 segments
[005/008] p030670.mat: 1764 segments
[006/008] p027884.mat: 1802 segments
[007/008] p043774.mat: 1943 segments
[008/008] p099008.mat: 1883 segments

Training segments: 14791
Input size: 1000
Train label mean [SBP, DBP]: [113.042465  64.16616 ]
Train label std  [SBP, DBP]: [11.188339   7.0302095]


In [7]:
# 6. Create one model
set_seed(SEED)

model, _ = select_model(
    backbone="sigma_ppg_pro",
    num_classes=2,
    in_chans=1,
    pretrained=True,
    checkpoint_path=CHECKPOINT_PATH,
    freeze_backbone_flag=True,
    device=DEVICE,
    patch_size=PATCH_SIZE,
    input_size=input_size,
)

model = model.to(DEVICE)


# ==============================================================================
# Partial fine-tuning 설정
# ==============================================================================

# 1. 우선 모든 parameter 동결
for parameter in model.parameters():
    parameter.requires_grad = False


# 2. 입력 projection, 마지막 normalization, regression head 학습 허용
TRAINABLE_KEYWORDS = [
    "patch_proj",
    "patch_embed",
    "fc_norm",
    "head",
]

for name, parameter in model.named_parameters():
    if any(keyword in name for keyword in TRAINABLE_KEYWORDS):
        parameter.requires_grad = True


# 3. 마지막 Transformer block 2개 학습 허용
NUM_UNFROZEN_BLOCKS = 2

if hasattr(model, "blocks"):
    for block in model.blocks[-NUM_UNFROZEN_BLOCKS:]:
        for parameter in block.parameters():
            parameter.requires_grad = True
else:
    print("Warning: model.blocks를 찾지 못했습니다.")


# ==============================================================================
# 실제 학습 parameter 확인
# ==============================================================================

print("\n" + "=" * 100)
print("TRAINABLE PARAMETERS")
print("=" * 100)

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(
            f"{name:75s} "
            f"{str(tuple(parameter.shape)):20s} "
            f"{parameter.numel():,}"
        )


trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_count = sum(
    parameter.numel()
    for parameter in trainable_parameters
)

print("\nTotal parameters    :", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_count:,}")
print(
    "Trainable ratio     :",
    f"{100 * trainable_count / total_parameters:.2f}%",
)


# ==============================================================================
# Optimizer parameter group
# ==============================================================================

head_parameters = []
backbone_parameters = []

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if "head" in name:
        head_parameters.append(parameter)
    else:
        backbone_parameters.append(parameter)


criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_parameters,
            "lr": 1e-5,
        },
        {
            "params": head_parameters,
            "lr": 1e-3,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

print("\nOptimizer groups")
print(
    "Backbone/adapter:",
    sum(p.numel() for p in backbone_parameters),
    "parameters, lr=1e-5",
)
print(
    "Regression head:",
    sum(p.numel() for p in head_parameters),
    "parameters, lr=1e-3",
)


🎯 Model Selection - Dynamic Configuration
Backbone: sigma_ppg_pro
Input size: 1000
Patch size: 50
Num classes: 2
In channels: 1
Pretrained: True

📦 Creating sigma_ppg_pro with dynamic parameters...

🏗️  Creating Model - Dynamic Configuration
Model size: sigma_ppg_pro
Input size: 1000
Patch size: 50
Number of patches: 20
Configuration:
  out_chans: 12
  embed_dim: 360
  depth: 18
  num_heads: 12
  drop_rate: 0.08
  attn_drop_rate: 0.08
  drop_path_rate: 0.15

✅ Model created with LayerScale (init_values=0.1)
✅ time_embed already correct size: torch.Size([1, 20, 360])
🔧 Resizing pos_embed: torch.Size([1, 21, 360]) -> torch.Size([1, 20, 360])

📦 Loading Pretrained Weights
   Path: /content/drive/MyDrive/Colab Notebooks/Mine/SIGMA-PPG/sigma.pth
   Strict shape match: False
✅ Found checkpoint['model']

📊 Checkpoint: 344 parameters
📊 Model: 365 parameters
  🔧 Interpolating time_embed: torch.Size([1, 120, 360]) -> torch.Size([1, 20, 360])
  🔧 Interpolating pos_embed: torch.Size([1, 129, 360]

In [8]:
# 7. Training and evaluation functions
def set_partial_finetune_mode(model):
    # 전체 모델은 기본적으로 evaluation mode
    model.eval()

    # 학습하는 입력 projection 계열
    for module_name in [
        "patch_proj",
        "patch_embed",
        "fc_norm",
        "head",
    ]:
        module = getattr(model, module_name, None)

        if module is not None:
            module.train()

    # 마지막 Transformer block들을 training mode로 설정
    if hasattr(model, "blocks"):
        for block in model.blocks[-NUM_UNFROZEN_BLOCKS:]:
            block.train()

def train_on_subject(model, mat_file):
    X_subject, y_subject = load_pulsedb_subject(mat_file)

    y_subject_normalized = (
        y_subject - y_mean
    ) / y_std

    loader = DataLoader(
        PulseDBDataset(
            X_subject,
            y_subject_normalized,
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

    set_partial_finetune_mode(model)

    total_loss = 0.0
    total_samples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        X_batch = rearrange(
            X_batch,
            "b c (n p) -> b c n p",
            p=PATCH_SIZE,
        )

        optimizer.zero_grad(set_to_none=True)

        prediction = model(X_batch)
        loss = criterion(prediction, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=1.0,
        )

        optimizer.step()

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    del X_subject, y_subject, y_subject_normalized, loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss, total_samples


def evaluate_subject(model, mat_file):
    X_subject, y_subject = load_pulsedb_subject(mat_file)

    y_subject_normalized = (
        y_subject - y_mean
    ) / y_std

    loader = DataLoader(
        PulseDBDataset(
            X_subject,
            y_subject_normalized,
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)

            X_batch = rearrange(
                X_batch,
                "b c (n p) -> b c n p",
                p=PATCH_SIZE,
            )

            prediction = model(X_batch)

            predictions.append(
                prediction.cpu().numpy()
            )
            targets.append(
                y_batch.numpy()
            )

    predictions = np.concatenate(
        predictions,
        axis=0,
    )
    targets = np.concatenate(
        targets,
        axis=0,
    )

    predictions_mmhg = (
        predictions * y_std + y_mean
    )
    targets_mmhg = (
        targets * y_std + y_mean
    )

    errors = predictions_mmhg - targets_mmhg

    result = {
        "Subject": mat_file.stem,
        "N": len(y_subject),
        "SBP_MAE": float(
            np.mean(np.abs(errors[:, 0]))
        ),
        "DBP_MAE": float(
            np.mean(np.abs(errors[:, 1]))
        ),
        "Mean_MAE": float(
            np.mean(np.abs(errors))
        ),
        "SBP_RMSE": float(
            np.sqrt(np.mean(errors[:, 0] ** 2))
        ),
        "DBP_RMSE": float(
            np.sqrt(np.mean(errors[:, 1] ** 2))
        ),
    }

    del X_subject, y_subject
    del y_subject_normalized, loader
    del predictions, targets
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [9]:
# 8. Train one model using all train subjects
for epoch in range(1, EPOCHS + 1):
    epoch_loss_sum = 0.0
    epoch_sample_count = 0

    # 매 epoch마다 train subject 순서를 섞음
    rng = np.random.default_rng(SEED + epoch)
    shuffled_files = list(train_files)
    rng.shuffle(shuffled_files)

    for subject_index, mat_file in enumerate(
        shuffled_files,
        start=1,
    ):
        subject_loss_sum, subject_count = (
            train_on_subject(
                model=model,
                mat_file=mat_file,
            )
        )

        epoch_loss_sum += subject_loss_sum
        epoch_sample_count += subject_count

    epoch_loss = (
        epoch_loss_sum / epoch_sample_count
    )

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] "
        f"Train MSE: {epoch_loss:.6f}"
    )

Epoch [001/150] Train MSE: 0.863705
Epoch [002/150] Train MSE: 0.618230
Epoch [003/150] Train MSE: 0.617833
Epoch [004/150] Train MSE: 0.580689
Epoch [005/150] Train MSE: 0.568199
Epoch [006/150] Train MSE: 0.502276
Epoch [007/150] Train MSE: 0.511957
Epoch [008/150] Train MSE: 0.490977
Epoch [009/150] Train MSE: 0.507661
Epoch [010/150] Train MSE: 0.639683
Epoch [011/150] Train MSE: 0.491602
Epoch [012/150] Train MSE: 0.471895
Epoch [013/150] Train MSE: 0.463200
Epoch [014/150] Train MSE: 0.427136
Epoch [015/150] Train MSE: 0.447140
Epoch [016/150] Train MSE: 0.354367
Epoch [017/150] Train MSE: 0.398109
Epoch [018/150] Train MSE: 0.406459
Epoch [019/150] Train MSE: 0.437968
Epoch [020/150] Train MSE: 0.416810
Epoch [021/150] Train MSE: 0.335846
Epoch [022/150] Train MSE: 0.336256
Epoch [023/150] Train MSE: 0.404522
Epoch [024/150] Train MSE: 0.296973
Epoch [025/150] Train MSE: 0.300574
Epoch [026/150] Train MSE: 0.330666
Epoch [027/150] Train MSE: 0.334019
Epoch [028/150] Train MSE: 0

In [10]:
# 9. Evaluate the one trained model on unseen test subjects
results = []
failed_subjects = []

for subject_index, mat_file in enumerate(
    test_files,
    start=1,
):
    print(
        f"[{subject_index:03d}/{len(test_files):03d}] "
        f"{mat_file.name}"
    )

    try:
        result = evaluate_subject(
            model=model,
            mat_file=mat_file,
        )

        results.append(result)

        print(
            f"SBP MAE: {result['SBP_MAE']:.3f} | "
            f"DBP MAE: {result['DBP_MAE']:.3f} | "
            f"Mean MAE: {result['Mean_MAE']:.3f} mmHg"
        )

    except Exception as error:
        failed_subjects.append(
            {
                "Subject": mat_file.stem,
                "Error": str(error),
            }
        )
        print("Skipped:", error)


if not results:
    raise RuntimeError(
        "No test subject was successfully evaluated."
    )


fieldnames = list(results[0].keys())

[001/002] p017795.mat
SBP MAE: 15.674 | DBP MAE: 10.183 | Mean MAE: 12.929 mmHg
[002/002] p098382.mat
SBP MAE: 25.828 | DBP MAE: 8.680 | Mean MAE: 17.254 mmHg


In [11]:
# 10. Unseen-subject summary
sbp_mae_values = np.asarray(
    [result["SBP_MAE"] for result in results],
    dtype=np.float64,
)

dbp_mae_values = np.asarray(
    [result["DBP_MAE"] for result in results],
    dtype=np.float64,
)

mean_mae_values = np.asarray(
    [result["Mean_MAE"] for result in results],
    dtype=np.float64,
)


def safe_std(values):
    if len(values) < 2:
        return 0.0

    return float(
        np.std(values, ddof=1)
    )


print("=" * 80)
print("One model, unseen-subject evaluation")
print("=" * 80)

print(
    "Average SBP MAE: "
    f"{np.mean(sbp_mae_values):.3f} ± "
    f"{safe_std(sbp_mae_values):.3f} mmHg"
)

print(
    "Average DBP MAE: "
    f"{np.mean(dbp_mae_values):.3f} ± "
    f"{safe_std(dbp_mae_values):.3f} mmHg"
)

print(
    "Overall average MAE: "
    f"{np.mean(mean_mae_values):.3f} ± "
    f"{safe_std(mean_mae_values):.3f} mmHg"
)

print("\nPer-subject results")
print(
    f"{'Subject':<15} {'N':>7} "
    f"{'SBP_MAE':>10} "
    f"{'DBP_MAE':>10} "
    f"{'Mean_MAE':>10}"
)

for result in results:
    print(
        f"{result['Subject']:<15} "
        f"{result['N']:>7d} "
        f"{result['SBP_MAE']:>10.3f} "
        f"{result['DBP_MAE']:>10.3f} "
        f"{result['Mean_MAE']:>10.3f}"
    )

if failed_subjects:
    print("\nFailed subjects")

    for failed in failed_subjects:
        print(
            f"- {failed['Subject']}: "
            f"{failed['Error']}"
        )

One model, unseen-subject evaluation
Average SBP MAE: 20.751 ± 7.180 mmHg
Average DBP MAE: 9.432 ± 1.063 mmHg
Overall average MAE: 15.091 ± 3.059 mmHg

Per-subject results
Subject               N    SBP_MAE    DBP_MAE   Mean_MAE
p017795            1726     15.674     10.183     12.929
p098382            1727     25.828      8.680     17.254
